# Apple music store — revenue questions in SQL

Chinook is a sample music-store database. This notebook answers three business questions against it, in SQL first, and then brings the answers into pandas.

**Ticket 1 — the map.** Before writing any query I need to know how the tables fit together and what one row of each table means. Nothing here answers a business question yet.

In [1]:
import sqlite3, pandas as pd
con = sqlite3.connect('data/Chinook_Sqlite.sqlite')
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


## Every table, and what one row means

| table | one row is |
|---|---|
| `Artist` | one artist |
| `Album` | one album, belonging to one artist |
| `Track` | one track, on one album, of one genre and media type |
| `Genre`, `MediaType` | lookup rows |
| `Playlist`, `PlaylistTrack` | a playlist, and one track's membership in one playlist |
| `Customer` | one customer, with a country, and the employee who supports them |
| `Employee` | one employee; `ReportsTo` points at their manager in the same table |
| `Invoice` | **one sale** — one customer, one date, one billing address, one total |
| `InvoiceLine` | **one track on one invoice** — unit price and quantity |

## Foreign keys, read from the database rather than guessed

SQLite stores the declared foreign keys. Reading them out is more reliable than inferring joins from column names.

In [2]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", con).name
rows = []
for t in tables:
    for fk in con.execute(f'PRAGMA foreign_key_list("{t}")'):
        rows.append({'from_table': t, 'from_column': fk[3], 'to_table': fk[2], 'to_column': fk[4]})
fks = pd.DataFrame(rows).sort_values(['from_table','from_column']).reset_index(drop=True)
fks

,from_table,from_column,to_table,to_column
0,Album,ArtistId,Artist,ArtistId
1,Customer,SupportRepId,Employee,EmployeeId
2,Employee,ReportsTo,Employee,EmployeeId
3,Invoice,CustomerId,Customer,CustomerId
4,InvoiceLine,InvoiceId,Invoice,InvoiceId
5,InvoiceLine,TrackId,Track,TrackId
6,PlaylistTrack,PlaylistId,Playlist,PlaylistId
7,PlaylistTrack,TrackId,Track,TrackId
8,Track,AlbumId,Album,AlbumId
9,Track,GenreId,Genre,GenreId


## The two join paths every later ticket needs

### Track → the country it was sold in

```
Track.TrackId  →  InvoiceLine.TrackId
InvoiceLine.InvoiceId  →  Invoice.InvoiceId
Invoice.CustomerId  →  Customer.CustomerId
```

That is **three hops**. There are two countries on the way: `Invoice.BillingCountry` (where this sale was billed) and `Customer.Country` (where the customer lives). They are usually the same but are separate columns, and ticket 2 has to say which one it uses.

### Track → the employee who owns that customer

```
… → Customer.SupportRepId  →  Employee.EmployeeId
```

One more hop past `Customer`. **`SupportRepId` is the only link from a sale to an employee** — there is no employee on `Invoice` itself, so "top-performing sales employee" can only mean *the rep whose customers spent the most*.

## Grain — the thing that decides whether a total is right

- **`InvoiceLine` is one row per track per invoice.** Counting rows here counts *tracks sold*; summing `UnitPrice * Quantity` here gives revenue per track.
- **`Invoice` is one row per sale.** Counting rows here counts *orders*; summing `Total` here gives revenue per invoice — and `Total` already equals the sum of that invoice's lines.

Why this matters: joining `Invoice` to `InvoiceLine` and then summing `Invoice.Total` counts each invoice's total **once per line on it** — an invoice with five tracks is counted five times. Revenue must be summed from `InvoiceLine`, or from `Invoice` alone, never from `Invoice.Total` after a join to lines.

In [3]:
# Sanity check on the grain claim: Invoice.Total really is the sum of its lines.
pd.read_sql_query('''
SELECT COUNT(*) AS invoices,
       SUM(CASE WHEN ROUND(i.Total, 2) = ROUND(l.line_total, 2) THEN 1 ELSE 0 END) AS totals_match_lines
FROM Invoice i
JOIN (SELECT InvoiceId, SUM(UnitPrice * Quantity) AS line_total FROM InvoiceLine GROUP BY InvoiceId) l
  ON l.InvoiceId = i.InvoiceId
''', con)

,invoices,totals_match_lines
0,412,412
